# k-Nearest Neighbours for Lemons vs Oranges

**Goal:** build and evaluate a $k$-NN classifier that decides whether a cropped fruit
patch is a lemon or an orange.

Object detection has two stages: localisation (where is the fruit?) and classification
(which fruit is it?). This notebook focuses on the classification stage. In a real
detector, a region-proposal step crops every candidate fruit into a small square patch;
the classifier then labels that patch. Here we work directly with such patches.

The workflow is the same as in the Iris, MNIST, and ImageNet companions:

1. load or generate the data;
2. split into train and test before preprocessing;
3. standardise the image features;
4. choose $k$ by cross-validation;
5. evaluate once on the held-out test set.

**Data options.** If `lemons_oranges/lemon` and `lemons_oranges/orange` folders exist next
to this notebook, their images are used. Otherwise the notebook generates a synthetic
dataset of yellow lemons and orange oranges, so it runs without a download.

## Setup

The notebook uses NumPy, matplotlib, and scikit-learn. Pillow is optional and is only
needed when loading a real image folder.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.decomposition import PCA
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
)
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

RANDOM_STATE = 42
IMAGE_SIZE = 32
MAX_PER_CLASS = 120
DATA_ROOT = Path('lemons_oranges')
IMAGE_EXTENSIONS = {'.png', '.jpg', '.jpeg', '.bmp', '.webp'}

np.random.seed(RANDOM_STATE)
plt.rcParams.update(
    {
        'figure.figsize': (8, 5),
        'axes.grid': True,
        'grid.alpha': 0.3,
        'axes.spines.top': False,
        'axes.spines.right': False,
    }
)
print('Setup complete.')

### Reusable helpers

`build_knn_pipeline` puts scaling, optional PCA, and the classifier in one object. Keeping
the scaler inside the pipeline ensures that cross-validation fits it on each training fold
only. `tune_k` returns the mean and standard deviation of the cross-validated accuracy for
each candidate $k$.

In [ ]:
def build_knn_pipeline(k, n_components=None):
    steps = [('scaler', StandardScaler())]
    if n_components is not None:
        steps.append(('pca', PCA(n_components=n_components, random_state=RANDOM_STATE)))
    steps.append(('knn', KNeighborsClassifier(n_neighbors=int(k), n_jobs=-1)))
    return Pipeline(steps)


def tune_k(X, y, ks, cv_splits=5):
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=RANDOM_STATE)
    rows = []
    for k in ks:
        scores = cross_val_score(build_knn_pipeline(k), X, y, cv=cv, scoring='accuracy')
        rows.append((int(k), float(scores.mean()), float(scores.std())))
    return rows


def plot_gallery(images, labels, class_names, n=12):
    n = min(n, len(images))
    cols = 6
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 1.7, rows * 1.8))
    axes = np.atleast_1d(axes).ravel()
    for ax, image, label in zip(axes, images[:n], labels[:n]):
        ax.imshow(image)
        ax.set_title(class_names[label], fontsize=9)
        ax.axis('off')
    for ax in axes[n:]:
        ax.axis('off')
    fig.tight_layout()
    plt.show()

## 1. Build or load the data

The synthetic generator draws lemons as yellow ellipses and oranges as orange circles on a
cluttered background. It varies position, size, orientation, brightness, highlights, and
noise so the two classes are not perfectly separable. A leaf appears in some images as a
distractor.

The folder loader accepts the usual ImageFolder layout:

```
lemons_oranges/
    lemon/
        img_0001.jpg
        ...
    orange/
        img_0001.jpg
        ...
```

Images are resized to 32 x 32 pixels and flattened, exactly as in the MNIST and ImageNet
companions.

In [ ]:
def draw_citrus_image(rng, kind, size=IMAGE_SIZE):
    yy, xx = np.mgrid[0:size, 0:size]
    x = (xx - (size - 1) / 2.0) / ((size - 1) / 2.0)
    y = (yy - (size - 1) / 2.0) / ((size - 1) / 2.0)

    background = rng.uniform(0.12, 0.40, size=3)
    image = np.ones((size, size, 3), dtype=np.float32) * background

    cx, cy = rng.uniform(-0.25, 0.25, size=2)
    angle = rng.uniform(-0.40, 0.40)
    cos_a, sin_a = np.cos(angle), np.sin(angle)
    xr = (x - cx) * cos_a + (y - cy) * sin_a
    yr = -(x - cx) * sin_a + (y - cy) * cos_a

    if kind == 0:
        a = rng.uniform(0.50, 0.68)
        b = rng.uniform(0.30, 0.43)
        mask = (xr / a) ** 2 + (yr / b) ** 2 <= 1.0
        base = np.array([0.96, 0.84, 0.10])
    else:
        radius = rng.uniform(0.42, 0.60)
        mask = xr ** 2 + yr ** 2 <= radius ** 2
        base = np.array([0.94, 0.45, 0.07])

    colour = np.clip(base * rng.uniform(0.80, 1.10, size=3), 0.0, 1.0)
    image[mask] = colour

    highlight = mask & ((((xr + 0.16) ** 2) / 0.020 + ((yr - 0.16) ** 2) / 0.020) <= 1.0)
    image[highlight] = np.clip(image[highlight] + 0.15, 0.0, 1.0)

    speckles = (rng.random((size, size)) < 0.03) & mask
    image[speckles] *= rng.uniform(0.85, 1.0)

    if rng.random() < 0.35:
        leaf_x = cx + rng.uniform(0.45, 0.70)
        leaf_y = cy + rng.uniform(0.35, 0.65)
        leaf = ((x - leaf_x) / 0.16) ** 2 + ((y - leaf_y) / 0.07) ** 2 <= 1.0
        image[leaf] = np.array([0.20, 0.55, 0.18])

    image += rng.normal(0.0, 0.035, size=image.shape)
    return np.clip(image, 0.0, 1.0)


def make_synthetic_citrus(n_per_class=MAX_PER_CLASS, image_size=IMAGE_SIZE, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    images, labels = [], []
    for kind in range(2):
        for _ in range(n_per_class):
            images.append(draw_citrus_image(rng, kind, image_size))
            labels.append(kind)
    images = np.stack(images).astype(np.float32)
    labels = np.asarray(labels, dtype=np.int64)
    order = rng.permutation(len(labels))
    return images[order], labels[order], ['lemon', 'orange']

### Optional folder loader

If you have your own cropped lemons-oranges patches, put them in the folder layout above.
The function below reads a balanced subset, resizes every image, and returns arrays with the
same shape as the synthetic data.

In [ ]:
def load_citrus_folder(root, image_size=IMAGE_SIZE, max_per_class=MAX_PER_CLASS, seed=RANDOM_STATE):
    from PIL import Image

    root = Path(root)
    class_names = ['lemon', 'orange']
    class_dirs = []
    for name in class_names:
        matches = [p for p in root.iterdir() if p.is_dir() and p.name.lower() == name]
        if not matches:
            raise FileNotFoundError(f'No {name} folder under {root}')
        class_dirs.append(matches[0])

    rng = np.random.default_rng(seed)
    images, labels = [], []
    for label, directory in enumerate(class_dirs):
        files = sorted(
            p for p in directory.iterdir()
            if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
        )
        if len(files) > max_per_class:
            pick = rng.choice(len(files), size=max_per_class, replace=False)
            files = [files[i] for i in sorted(pick)]
        for path in files:
            with Image.open(path) as image:
                array = np.asarray(
                    image.convert('RGB').resize((image_size, image_size)),
                    dtype=np.float32,
                ) / 255.0
            images.append(array)
            labels.append(label)

    if not images:
        raise FileNotFoundError(f'No images found under {root}')

    images = np.stack(images).astype(np.float32)
    labels = np.asarray(labels, dtype=np.int64)
    counts = np.bincount(labels, minlength=len(class_names))
    if counts.min() < 2:
        raise ValueError(f'Need at least two images per class under {root}')
    order = rng.permutation(len(labels))
    return images[order], labels[order], class_names

### Load the images

We try the real folder first and fall back to synthetic images only when it is missing or
unreadable. The printed source tells you which path was taken.

In [ ]:
try:
    X_img, y_img, class_names = load_citrus_folder(DATA_ROOT)
    source = f'images from {DATA_ROOT.resolve()}'
except Exception as exc:
    print(f'Using synthetic data ({type(exc).__name__}: {exc})')
    X_img, y_img, class_names = make_synthetic_citrus()
    source = 'synthetic lemons and oranges'

print(f'Source: {source}')
print(f'Images: {X_img.shape} -> {X_img.shape[1] * X_img.shape[2] * X_img.shape[3]} raw features')
print(f'Classes: {class_names}')
print('Class counts:', np.bincount(y_img))

for label, name in enumerate(class_names):
    mean_rgb = X_img[y_img == label].mean(axis=(0, 1, 2))
    print(f'{name:6s} mean RGB = {np.round(mean_rgb, 3)}')

plot_gallery(X_img, y_img, class_names, n=12)

## 2. Split first, then scale

The split comes before any preprocessing. The scaler is fitted on the training set only,
and it lives inside the pipeline so cross-validation repeats that rule on every fold. This
prevents the test set from leaking into preprocessing.

In [ ]:
X_img_train, X_img_test, y_train, y_test = train_test_split(
    X_img, y_img, test_size=0.30, stratify=y_img, random_state=RANDOM_STATE
)
X_train = X_img_train.reshape(len(X_img_train), -1)
X_test = X_img_test.reshape(len(X_img_test), -1)

print(f'Train: {X_train.shape}  Test: {X_test.shape}')

baseline = build_knn_pipeline(5).fit(X_train, y_train)
print(f'5-NN test accuracy (no tuning): {baseline.score(X_test, y_test):.3f}')

## 3. Choose $k$ with cross-validation

We try odd values from 1 to 31 with stratified 5-fold cross-validation. Odd values avoid
ties in a binary vote. The test set stays untouched until the next section.

In [ ]:
ks = list(range(1, 32, 2))
rows = tune_k(X_train, y_train, ks, cv_splits=5)
best_k, best_mean, best_std = max(rows, key=lambda row: row[1])

fig, ax = plt.subplots()
ax.errorbar([row[0] for row in rows], [row[1] for row in rows],
            yerr=[row[2] for row in rows], marker='o', capsize=3)
ax.axvline(best_k, color='k', linestyle=':', label=f'best k = {best_k}')
ax.set(xlabel='$k$', ylabel='Cross-validated accuracy',
       title='Lemon vs orange: choosing k')
ax.legend()
plt.show()

for k, mean, std in rows:
    print(f'k = {k:2d}   CV accuracy = {mean:.3f} +/- {std:.3f}')
print(f'Selected k = {best_k} with CV accuracy {best_mean:.3f} +/- {best_std:.3f}')

## 4. Final model and test set

Refit the selected $k$ on the full training set and evaluate once on the held-out test set.
This is the number to report.

In [ ]:
final_model = build_knn_pipeline(best_k).fit(X_train, y_train)
y_pred = final_model.predict(X_test)

print(f'k = {best_k}')
print(f'Test accuracy: {accuracy_score(y_test, y_pred):.3f}')
print()
print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
    ax=ax, cmap='Oranges', colorbar=False
)
ax.set_title(f'Lemon vs orange confusion matrix (k = {best_k})')
plt.show()

## 5. What the model sees: a two-dimensional PCA view

The final model uses all pixel features. A PCA projection is useful for visualisation: it
compresses each patch to two coordinates, and we can draw the $k$-NN decision regions in
that plane. This plot is a visual aid, not the final feature space, so its accuracy can
differ from the full model.

In [ ]:
scaler = StandardScaler().fit(X_train)
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_train_2d = pca.fit_transform(scaler.transform(X_train))
X_test_2d = pca.transform(scaler.transform(X_test))
print(f'Variance explained by two components: {pca.explained_variance_ratio_.sum():.3f}')

viz_model = KNeighborsClassifier(n_neighbors=best_k).fit(X_train_2d, y_train)
x_min, x_max = X_train_2d[:, 0].min() - 0.5, X_train_2d[:, 0].max() + 0.5
y_min, y_max = X_train_2d[:, 1].min() - 0.5, X_train_2d[:, 1].max() + 0.5
xx, yy = np.meshgrid(
    np.linspace(x_min, x_max, 400),
    np.linspace(y_min, y_max, 400),
)
zz = viz_model.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots()
ax.contourf(xx, yy, zz, levels=[-0.5, 0.5, 1.5],
            colors=['#FFF3B0', '#F4A261'], alpha=0.65)
styles = [
    (0, 'lemon', 'o', '#E9C46A'),
    (1, 'orange', 's', '#E76F51'),
]
for label, name, marker, colour in styles:
    train_mask = y_train == label
    test_mask = y_test == label
    ax.scatter(X_train_2d[train_mask, 0], X_train_2d[train_mask, 1],
               marker=marker, s=35, color=colour, edgecolor='k',
               label=f'{name} train')
    ax.scatter(X_test_2d[test_mask, 0], X_test_2d[test_mask, 1],
               marker='x', s=55, color=colour, label=f'{name} test')
ax.set(xlabel='PC 1', ylabel='PC 2',
       title=f'PCA view and k-NN regions (k = {best_k})')
ax.legend()
plt.show()

print(f'Two-component k-NN test accuracy: {viz_model.score(X_test_2d, y_test):.3f}')

## Where this fits in a detection system

In a full detector, a region-proposal model supplies bounding boxes. For each box we crop,
resize, flatten, and apply the classifier above. The main lesson is that $k$-NN can work as a
simple baseline, but raw pixels are sensitive to translation, scale, and background. If you
move on to a real lemons-oranges detection dataset, keep the same train/test discipline and
consider colour histograms, HOG features, or CNN embeddings before $k$-NN.

**Takeaways**

- split before scaling; the scaler belongs in the pipeline;
- tune $k$ by cross-validation, then report one held-out test score;
- inspect the confusion matrix, not only accuracy;
- the PCA plot is a visual aid, not the final feature space;
- for object detection, k-NN classifies proposed patches, while a separate model localises them.

## What to try next

1. **Colour features.** Replace raw pixels with per-channel histograms and compare accuracy.
2. **Distance weighting.** Try `weights='distance'` in `KNeighborsClassifier`.
3. **Real data.** Put cropped lemon and orange patches in `lemons_oranges/lemon` and
   `lemons_oranges/orange`, restart, and rerun the notebook.
4. **Augmentation.** Flip, rotate, and crop the training patches, then retrain. Does the
   decision boundary become more stable?
5. **Error analysis.** Display the misclassified test images and look for leaves, shadows,
   or unusual fruit sizes.

## Further reading

- T. Cover and P. Hart, 'Nearest Neighbor Pattern Classification', IEEE Transactions on
  Information Theory, 1967.
- scikit-learn user guide, 'Nearest Neighbors'.
- Any introductory treatment of the bias-variance trade-off for $k$-NN.